In [1]:
"""
Step 3 (MONTHLY variant): Granger causality test on the monthly-resampled
exit-intention / CVE series, fixing the shared-trend AND shared-seasonal-
pattern problems the same way the daily/weekly version does, just at
monthly granularity.

Workflow:
  1. DESEASONALIZE: regress each series on MONTH-OF-YEAR dummies (Jan-Dec)
     and keep the residuals. This replaces the day-of-week + Patch-Tuesday
     deseasonalization used in the daily/weekly script -- there is no
     "day of week" at monthly granularity, but there can still be a real
     annual pattern (e.g. budget-cycle-driven CVE disclosure clusters,
     back-to-school/new-year burnout posting waves) that would otherwise
     look like cross-series "causality." No Patch-Tuesday indicator here;
     that's a weekly-cadence effect that monthly aggregation already
     absorbs.
  2. ADF unit-root test PLUS an explicit linear-trend test on the
     deseasonalized residuals, exactly as in the daily/weekly script.
  3. If EITHER check flags a problem, difference BOTH series in a pair by
     the same order.
  4. VAR order selection (AIC and BIC), capped at MAX_LAG_FOR_ORDER_SELECTION
     = 12 (one year of monthly lags) instead of 84 -- 84 monthly lags would
     be 7 years, which both overfits and eats nearly the whole 2018-2026
     sample just for lag order selection.
  5. Granger causality test at both selected lags, in BOTH directions,
     same as before.

Reads the CSV from build_q2_monthly_timeseries.py. No raw file access here.
This is a new, standalone script -- it does not modify granger_q2_v1.ipynb
or any daily/weekly script.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
import os
import warnings

warnings.filterwarnings("ignore")  # statsmodels VAR is chatty about lag-order edge cases

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q2_analysis/"
MONTHLY_CSV = os.path.join(OUT_DIR, "exit_monthly_series.csv")

EXIT_COLS = ["n_exit_any", "n_exit_explicit"]
CVE_COLS = ["n_cve_run1", "n_cve_run2"]

MAX_LAG_FOR_ORDER_SELECTION = 12  # was 84 in the daily/weekly script -- one year of monthly lags
MAX_DIFF_ORDER = 2
ALPHA = 0.05


def deseasonalize(monthly_df, date_col, value_cols):
    """Regress each value column on month-of-year dummies (Jan-Dec, one
    dropped as baseline). Returns a DataFrame of residuals (same index/order
    as input) plus prints the R^2 for each column so you can see how much
    annual seasonal pattern was actually present."""
    dates = pd.to_datetime(monthly_df[date_col])
    month = dates.dt.month

    month_dummies = pd.get_dummies(month, prefix="month", drop_first=True)
    X = sm.add_constant(month_dummies).astype(float)

    residuals = {}
    print("  Month-of-year deseasonalization:")
    for col in value_cols:
        y = monthly_df[col].astype(float)
        model = sm.OLS(y, X).fit()
        residuals[col] = model.resid
        print(f"    {col}: R^2 = {model.rsquared:.4f} "
              f"({'meaningful seasonal pattern present' if model.rsquared > 0.02 else 'little/no seasonal pattern'})")

    return pd.DataFrame(residuals)


def adf_pvalue(series):
    if series.std() == 0:
        return 1.0  # constant series -- treat as "non-stationary" (forces a difference attempt,
                     # which will surface a clearer error if the series is genuinely too sparse)
    return adfuller(series.dropna(), autolag="AIC")[1]


def trend_pvalue_and_slope(series):
    """OLS regression of the series against a plain time index (0,1,2,...).
    Returns (p_value, slope) for the slope coefficient. A significant
    p-value means a real deterministic trend exists over the period, even
    if the ADF unit-root test alone doesn't flag it."""
    s = series.dropna().values
    t = np.arange(len(s))
    slope, intercept, r, p, se = scipy_stats.linregress(t, s)
    return p, slope


def find_common_diff_order(series_a, series_b, label_a, label_b, max_d=MAX_DIFF_ORDER):
    """Difference both series together, by the same number of steps, until
    BOTH pass the unit-root test AND show no significant linear trend (or
    until max_d is hit)."""
    a, b = series_a.copy(), series_b.copy()
    d = 0
    while d <= max_d:
        p_a_unit, p_b_unit = adf_pvalue(a), adf_pvalue(b)
        p_a_trend, slope_a = trend_pvalue_and_slope(a)
        p_b_trend, slope_b = trend_pvalue_and_slope(b)

        unit_a = "no unit root" if p_a_unit < ALPHA else "UNIT ROOT"
        unit_b = "no unit root" if p_b_unit < ALPHA else "UNIT ROOT"
        trend_a = f"TREND (slope={slope_a:.5f}, p={p_a_trend:.4g})" if p_a_trend < ALPHA else "no significant trend"
        trend_b = f"TREND (slope={slope_b:.5f}, p={p_b_trend:.4g})" if p_b_trend < ALPHA else "no significant trend"

        print(f"    d={d}: {label_a}: ADF p={p_a_unit:.4g} ({unit_a}); linear trend check: {trend_a}")
        print(f"           {label_b}: ADF p={p_b_unit:.4g} ({unit_b}); linear trend check: {trend_b}")

        a_ok = (p_a_unit < ALPHA) and (p_a_trend >= ALPHA)
        b_ok = (p_b_unit < ALPHA) and (p_b_trend >= ALPHA)

        if a_ok and b_ok:
            return a, b, d

        a = a.diff().dropna()
        b = b.diff().dropna()
        d += 1
    print(f"    WARNING: still non-stationary/trending after d={max_d} -- proceeding anyway, "
          f"interpret results cautiously")
    return a, b, d


def select_lag_orders(df_pair, maxlags):
    model = VAR(df_pair)
    sel = model.select_order(maxlags=maxlags)
    return sel


def granger_at_lag(effect_series, cause_series, lag, direction_label):
    """data columns must be [effect, cause] -- statsmodels tests whether
    column 1 (cause) Granger-causes column 0 (effect)."""
    df_pair = pd.DataFrame({"effect": effect_series.values, "cause": cause_series.values})
    results = grangercausalitytests(df_pair, maxlag=[lag], verbose=False)
    fstat, pvalue, df_denom, df_num = results[lag][0]["ssr_ftest"]
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"
    print(f"    {direction_label} at lag={lag}: F={fstat:.4f}, p={pvalue:.4g}  -> {sig} (alpha={ALPHA})")
    return fstat, pvalue


def main():
    if not os.path.exists(MONTHLY_CSV):
        print(f"Could not find {MONTHLY_CSV}. Run build_q2_monthly_timeseries.py first.")
        return

    monthly = pd.read_csv(MONTHLY_CSV)
    print(f"Loaded {len(monthly)} months from {MONTHLY_CSV}")

    n_lags_possible = len(monthly) - 1
    if MAX_LAG_FOR_ORDER_SELECTION > n_lags_possible // 3:
        print(f"  NOTE: {len(monthly)} months is a fairly short series for lag-order selection "
              f"up to {MAX_LAG_FOR_ORDER_SELECTION} -- interpret high-lag results cautiously.")

    print(f"\n{'=' * 70}")
    print("DESEASONALIZING (month-of-year)")
    print("=" * 70)
    deseasonalized = deseasonalize(monthly, "date", EXIT_COLS + CVE_COLS)

    summary_rows = []

    for exit_col in EXIT_COLS:
        for cve_col in CVE_COLS:
            print(f"\n{'=' * 70}")
            print(f"{exit_col}  vs  {cve_col}")
            print("=" * 70)

            print("\n  Stationarity check + common differencing order (on deseasonalized residuals):")
            exit_s, cve_s, d = find_common_diff_order(
                deseasonalized[exit_col], deseasonalized[cve_col], exit_col, cve_col
            )
            print(f"  Using d={d} for both series in this pair")

            df_pair = pd.DataFrame({exit_col: exit_s.values, cve_col: cve_s.values})

            print(f"\n  Selecting lag order (AIC/BIC, up to {MAX_LAG_FOR_ORDER_SELECTION} months):")
            sel = select_lag_orders(df_pair, MAX_LAG_FOR_ORDER_SELECTION)
            aic_lag = max(int(sel.aic), 1)
            bic_lag = max(int(sel.bic), 1)
            print(f"  AIC-selected lag: {aic_lag}   |   BIC-selected lag: {bic_lag}")

            print(f"\n  Granger causality -- does CVE lead exit intention?")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(exit_s, cve_s, lag, f"CVE -> exit intention ({crit_name} lag)")
                summary_rows.append({
                    "exit_threshold": exit_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "cve_to_exit", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

            print(f"\n  Reverse check -- does exit intention lead CVE? (should NOT be significant for a clean story)")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(cve_s, exit_s, lag, f"exit intention -> CVE ({crit_name} lag)")
                summary_rows.append({
                    "exit_threshold": exit_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "exit_to_cve", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

    summary_df = pd.DataFrame(summary_rows)
    out_path = os.path.join(OUT_DIR, "exit_granger_causality_results(monthly).csv")
    summary_df.to_csv(out_path, index=False)

    print(f"\n{'=' * 70}")
    print("FULL SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")

    print(f"\n{'=' * 70}")
    print("HOW TO READ THIS")
    print("=" * 70)
    print("Same rule as the daily/weekly version: the bar for a real finding is")
    print("direction='cve_to_exit' significant (p < 0.05) AND the matching")
    print("direction='exit_to_cve' row (same combination + criterion) NOT significant.")
    print("\nIf exit_to_cve is significant ANYWHERE, that specific row is not trustworthy")
    print("evidence for your hypothesis -- something is still shared between the two")
    print("series that hasn't been fully removed (annual seasonal pattern, trend, or")
    print("some other confound).")
    print("\nWeight BIC rows more than AIC rows when they disagree, same reasoning as")
    print("before -- AIC tends to pick larger, more overfit-prone lags.")
    print("\nA caution specific to the monthly version: with ~101 months (2018-2026),")
    print("a 12-month max lag for order selection uses a much larger share of the")
    print("sample than 84 days did out of ~3000+ days. If AIC/BIC select lags near")
    print("the 12-month ceiling, treat that as a sign the monthly series may be too")
    print("short to support that much lag structure reliably, not as a strong finding.")


if __name__ == "__main__":
    main()

Loaded 101 months from /Users/nadia/Desktop/redditRun_june/q2_analysis/exit_monthly_series.csv

DESEASONALIZING (month-of-year)
  Month-of-year deseasonalization:
    n_exit_any: R^2 = 0.0525 (meaningful seasonal pattern present)
    n_exit_explicit: R^2 = 0.0960 (meaningful seasonal pattern present)
    n_cve_run1: R^2 = 0.0385 (meaningful seasonal pattern present)
    n_cve_run2: R^2 = 0.1388 (meaningful seasonal pattern present)

n_exit_any  vs  n_cve_run1

  Stationarity check + common differencing order (on deseasonalized residuals):
    d=0: n_exit_any: ADF p=0.5873 (UNIT ROOT); linear trend check: TREND (slope=1.15192, p=5.865e-24)
           n_cve_run1: ADF p=0.1372 (UNIT ROOT); linear trend check: TREND (slope=-1.18274, p=1.943e-06)
    d=1: n_exit_any: ADF p=0.01682 (no unit root); linear trend check: no significant trend
           n_cve_run1: ADF p=2.471e-09 (no unit root); linear trend check: no significant trend
  Using d=1 for both series in this pair

  Selecting lag or